# Practica 2 - Pipeline de 2 pasos con LLMs para resenas de videojuegos

Este notebook implementa un pipeline de dos pasos con LLMs:

1. **Paso 1 (filtro):** clasifica resenas como relevantes o no relevantes.
2. **Paso 2 (extraccion):** extrae campos estructurados de las resenas filtradas.

Se usa **Gemini API** (Google) por buen equilibrio entre calidad, latencia y facilidad para procesar lotes.
Tambien se documenta como se controlan limitaciones: tokens por llamada, rate limits, tamano de lote, pausas entre llamadas y reintentos con backoff.

## Diseno y justificacion tecnica

**Proveedor/modelo elegido:** Google Gemini + `gemini-3.8-flash`

- Motivo: buen equilibrio coste/velocidad para clasificacion y extraccion en lotes.
- Paso 1 usa lotes mas grandes porque solo devuelve etiquetas breves.
- Paso 2 usa lotes mas pequenos porque devuelve JSON mas largo por resena.

**Gestion de limitaciones**

- Se estima longitud de entrada por lote para no exceder contexto.
- Se configura `max_tokens` diferente por paso.
- Se usa `time.sleep` entre llamadas para suavizar trafico.
- Se aplica reintento con backoff exponencial ante fallos transitorios.
- Se guardan checkpoints por lote para poder reanudar si se agota cuota o hay error de red.
- Se aplica fallback de modelo si el proveedor depreca el modelo principal.

In [25]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [39]:
# -------------------------------
# Configuracion global del pipeline
# -------------------------------
MODEL_FILTER = "gemini-3.8-flash"
MODEL_EXTRACT = "gemini-3.8-flash"
FALLBACK_MODELS = ["gemini-2.5-flash", "gemini-2.5-pro"]

BATCH_SIZE_FILTER = 5
BATCH_SIZE_EXTRACT = 1

MAX_TOKENS_FILTER = 2048
MAX_TOKENS_EXTRACT = 2200

MAX_CHARS_PER_REVIEW_FILTER = 1400
MAX_CHARS_PER_REVIEW_EXTRACT = 1200

SLEEP_BETWEEN_CALLS = 1.2
MAX_RETRIES = 4
BACKOFF_BASE_SECONDS = 2

TOP_N_LONGEST_REVIEWS = 100

In [27]:
def find_project_root(start: Path) -> Path:
    """Busca la raiz del proyecto subiendo directorios."""
    current = start.resolve()
    for path in [current, *current.parents]:
        candidate = path / "data" / "raw" / "videogames_reviews.csv"
        if candidate.exists():
            return path
    raise FileNotFoundError("No se encontro data/raw/videogames_reviews.csv")


def chunked(items: List[Dict[str, Any]], batch_size: int) -> Iterable[List[Dict[str, Any]]]:
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]


def estimate_tokens(text: str) -> int:
    """Estimacion simple: ~4 caracteres por token en promedio."""
    return max(1, len(text) // 4)


def _strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        lines = text.splitlines()
        if len(lines) >= 3:
            return "\n".join(lines[1:-1]).strip()
    return text


def _extract_list_from_dict(data: Dict[str, Any]) -> List[Dict[str, Any]] | None:
    for key in ["items", "results", "data", "reviews", "output"]:
        value = data.get(key)
        if isinstance(value, list):
            return value
    return None


def _try_parse_json_value(text: str, start_idx: int) -> tuple[Any, int] | None:
    decoder = json.JSONDecoder()
    try:
        value, end_idx = decoder.raw_decode(text[start_idx:])
        return value, start_idx + end_idx
    except json.JSONDecodeError:
        return None


def extract_json_array(raw_text: str) -> List[Dict[str, Any]]:
    """Parsea respuestas JSON en varios formatos comunes del LLM."""
    cleaned = _strip_code_fences((raw_text or "").strip())
    if not cleaned:
        raise ValueError("Respuesta vacia del modelo")

    # 1) Intento directo
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            extracted = _extract_list_from_dict(data)
            if extracted is not None:
                return extracted
    except json.JSONDecodeError:
        pass

    # 2) Escaneo robusto para encontrar el primer JSON valido
    for idx, char in enumerate(cleaned):
        if char not in "[{":
            continue
        parsed = _try_parse_json_value(cleaned, idx)
        if not parsed:
            continue

        value, _ = parsed
        if isinstance(value, list):
            return value
        if isinstance(value, dict):
            extracted = _extract_list_from_dict(value)
            if extracted is not None:
                return extracted

    preview = cleaned[:600].replace("\n", " ")
    raise ValueError(
        f"No se pudo parsear JSON tipo lista desde la respuesta del modelo. Preview: {preview}"
    )


def _is_model_not_found_error(exc: Exception) -> bool:
    text = str(exc).lower()
    return ("404" in text or "not_found" in text) and ("model" in text)


def call_llm_with_retry(
    client: genai.Client,
    model: str,
    system_prompt: str,
    user_prompt: str,
    max_tokens: int,
    temperature: float = 0.0,
) -> str:
    """Llama al modelo con reintentos, backoff y fallback de modelo."""
    candidate_models = [model, *FALLBACK_MODELS]
    last_error: Exception | None = None

    for chosen_model in candidate_models:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = client.models.generate_content(
                    model=chosen_model,
                    contents=user_prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=system_prompt,
                        temperature=temperature,
                        max_output_tokens=max_tokens,
                        response_mime_type="application/json",
                    ),
                )
                if chosen_model != model:
                    print(f"Usando fallback de modelo: {chosen_model}")
                return response.text or ""
            except Exception as exc:
                last_error = exc
                if _is_model_not_found_error(exc):
                    print(f"Modelo no disponible: {chosen_model}. Probando siguiente fallback...")
                    break

                wait_seconds = BACKOFF_BASE_SECONDS ** attempt
                print(f"Intento {attempt}/{MAX_RETRIES} fallo con {chosen_model}: {exc}")
                if attempt < MAX_RETRIES:
                    print(f"Esperando {wait_seconds}s antes de reintentar...")
                    time.sleep(wait_seconds)

    raise RuntimeError("Fallo en todos los modelos configurados tras reintentos") from last_error


project_root = find_project_root(Path.cwd())
load_dotenv(project_root / ".env")

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise EnvironmentError(
        "Define GEMINI_API_KEY en .env para ejecutar el pipeline con Gemini API"
    )

client = genai.Client(api_key=gemini_api_key)
print("Cliente Gemini inicializado correctamente.")

Cliente Gemini inicializado correctamente.


In [28]:
# -------------------------------
# 1) Carga y exploracion basica
# -------------------------------
import re
import unicodedata


def normalize_column_name(name: str) -> str:
    """Normaliza nombres de columna para evitar fallos por tildes/espacios."""
    normalized = unicodedata.normalize("NFKD", str(name))
    normalized = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    normalized = normalized.replace("\ufeff", "")
    normalized = re.sub(r"\s+", "_", normalized.strip())
    return normalized


csv_path = project_root / "data" / "raw" / "videogames_reviews.csv"
df = pd.read_csv(csv_path)

# Estandarizar cabeceras: p.ej. Valoración -> Valoracion
df.columns = [normalize_column_name(col) for col in df.columns]

# Eliminar columna indice exportada por pandas si existe
if "Unnamed:_0" in df.columns:
    df = df.drop(columns=["Unnamed:_0"]).copy()

print("Forma del dataframe:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDistribucion de 'Valoracion':")
print(df["Valoracion"].value_counts(dropna=False))

df.head(3)

Forma del dataframe: (20000, 3)

Columnas: ['Contenido', 'Valoracion', 'Recomendado_binario']

Valores nulos por columna:
Contenido              288
Valoracion               0
Recomendado_binario      0
dtype: int64

Distribucion de 'Valoracion':
Valoracion
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64


,Contenido,Valoracion,Recomendado_binario
0,2 marzo so bad,No recomendado,0
1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0


In [29]:
# -------------------------------
# 2) Preprocesamiento
# -------------------------------
df_clean = df.copy()

# Eliminar filas con contenido nulo o vacio
df_clean = df_clean.dropna(subset=["Contenido"])
df_clean["Contenido"] = df_clean["Contenido"].astype(str).str.strip()
df_clean = df_clean[df_clean["Contenido"] != ""].copy()

# Crear longitud para muestreo representativo
df_clean["longitud_texto"] = df_clean["Contenido"].str.len()

# Seleccionar los N comentarios mas largos
df_sample = (
    df_clean.sort_values("longitud_texto", ascending=False)
    .head(TOP_N_LONGEST_REVIEWS)
    .copy()
)

# ID estable para trazabilidad y checkpoints
df_sample = df_sample.reset_index(drop=False).rename(columns={"index": "review_id"})

print("Filas tras limpieza:", len(df_clean))
print("Tamano de muestra seleccionada:", len(df_sample))
df_sample[["review_id", "longitud_texto", "Valoracion"]].head()

Filas tras limpieza: 19712
Tamano de muestra seleccionada: 100


,review_id,longitud_texto,Valoracion
0,19502,7972,Recomendado
1,17660,7662,Recomendado
2,7187,7638,No recomendado
3,4187,7638,No recomendado
4,7751,7609,No recomendado


## Paso 1 - Filtrado de resenas irrelevantes con LLM

En este paso, el modelo decide si cada resena es relevante para analisis de experiencia de juego.

Se considera **no relevante** si contiene spam, ruido, muy poco contenido evaluable, texto repetitivo, o mensajes que no describen experiencia/opinion del juego.

Salida esperada por resena:
- `review_id`
- `is_relevant` (true/false)
- `reason` (explicacion breve)

In [30]:
def build_filter_prompt(batch_rows: List[Dict[str, Any]]) -> str:
    rows_json = json.dumps(batch_rows, ensure_ascii=False, indent=2)
    return f"""
Clasifica cada resena como relevante o no relevante para analisis de videojuegos.

Reglas:
- Relevante: describe experiencia, opinion, calidad, rendimiento, jugabilidad, historia, precio, etc.
- No relevante: spam, insultos sin contexto, texto vacio, solo emojis, texto incomprensible, ruido.

Devuelve SOLO un JSON valido (lista de objetos), sin texto adicional.
Cada objeto debe tener:
- review_id (int)
- is_relevant (bool)
- reason (string corto, max 12 palabras)

Datos de entrada:
{rows_json}
"""


def run_filter_step(
    df_input: pd.DataFrame,
    checkpoint_path: Path,
    resume: bool = True,
) -> pd.DataFrame:
    records = df_input[["review_id", "Contenido"]].to_dict(orient="records")

    # Recorte defensivo para evitar prompts gigantes en resenas muy largas
    for row in records:
        text = str(row.get("Contenido", ""))
        row["Contenido"] = text[:MAX_CHARS_PER_REVIEW_FILTER]

    done_map: Dict[int, Dict[str, Any]] = {}
    if resume and checkpoint_path.exists():
        previous = pd.read_json(checkpoint_path, lines=True)
        for _, row in previous.iterrows():
            done_map[int(row["review_id"])] = row.to_dict()
        print(f"Checkpoint encontrado con {len(done_map)} resenas ya procesadas.")

    pending_records = [r for r in records if int(r["review_id"]) not in done_map]
    print(f"Resenas pendientes de filtrar: {len(pending_records)}")

    system_prompt = (
        "Eres un analista de NLP. Devuelves solo JSON valido y consistente."
    )

    for batch_number, batch in enumerate(chunked(pending_records, BATCH_SIZE_FILTER), start=1):
        total_chars = sum(len(item["Contenido"]) for item in batch)
        estimated_in_tokens = estimate_tokens(str(batch))
        print(
            f"Lote filtro {batch_number}: {len(batch)} resenas | "
            f"chars={total_chars} | est_tokens_in={estimated_in_tokens}"
        )

        prompt = build_filter_prompt(batch)
        raw_response = call_llm_with_retry(
            client=client,
            model=MODEL_FILTER,
            system_prompt=system_prompt,
            user_prompt=prompt,
            max_tokens=MAX_TOKENS_FILTER,
            temperature=0.0,
        )

        parsed = extract_json_array(raw_response)
        if len(parsed) != len(batch):
            print("Aviso: numero de salidas distinto al tamano del lote.")

        for item in parsed:
            rid = int(item["review_id"])
            done_map[rid] = {
                "review_id": rid,
                "is_relevant": bool(item.get("is_relevant", False)),
                "reason": str(item.get("reason", "")),
            }

        pd.DataFrame(done_map.values()).to_json(
            checkpoint_path, orient="records", lines=True, force_ascii=False
        )
        time.sleep(SLEEP_BETWEEN_CALLS)

    filtered_df = pd.DataFrame(done_map.values()).sort_values("review_id")
    return filtered_df

In [31]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

filter_checkpoint = processed_dir / "checkpoint_step1_filter.jsonl"
df_filter_results = run_filter_step(df_sample, filter_checkpoint, resume=True)

df_step1 = df_sample.merge(df_filter_results, on="review_id", how="left")
df_step1_relevant = df_step1[df_step1["is_relevant"] == True].copy()

print("Total muestra entrada:", len(df_sample))
print("Resenas relevantes tras Paso 1:", len(df_step1_relevant))
print("Resenas descartadas:", len(df_sample) - len(df_step1_relevant))

df_step1[["review_id", "is_relevant", "reason", "Valoracion"]].head(10)

Checkpoint encontrado con 100 resenas ya procesadas.
Resenas pendientes de filtrar: 0
Total muestra entrada: 100
Resenas relevantes tras Paso 1: 99
Resenas descartadas: 1


,review_id,is_relevant,reason,Valoracion
0,19502,False,Spam repetitivo y texto incomprensible sin ana...,Recomendado
1,17660,True,"Analiza detalladamente el diseno del mundo, pr...",Recomendado
2,7187,True,"Critica estructurada sobre narrativa, expectat...",No recomendado
3,4187,True,"Critica estructurada sobre narrativa, expectat...",No recomendado
4,7751,True,"Analisis exhaustivo de jugabilidad, diseno de ...",No recomendado
5,5415,True,"Analiza dificultad, diseno de enemigos, combat...",No recomendado
6,16503,True,"Detalla experiencia previa, mecanicas de camin...",Recomendado
7,7805,True,"Critica funciones recortadas, inteligencia art...",No recomendado
8,13941,True,"Evalua combate, rendimiento tecnico, dificulta...",Recomendado
9,12814,True,"Evalua combate, rendimiento tecnico, dificulta...",Recomendado


In [32]:
# Guardado de resultados intermedios del paso 1
step1_json_path = processed_dir / "ej1_parte1_entidades.json"
df_step1.to_json(step1_json_path, orient="records", force_ascii=False, indent=2)
print(f"Guardado Paso 1 en: {step1_json_path}")

df_step1[df_step1["is_relevant"] == False][["review_id", "reason", "Contenido"]].head(5)

Guardado Paso 1 en: C:\Users\venel\Documents\GitHub\IA0526\PRACTICA2_PROMPT\data\processed\ej1_parte1_entidades.json


,review_id,reason,Contenido
0,19502,Spam repetitivo y texto incomprensible sin ana...,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...


## Paso 2 - Extraccion estructurada con LLM

Sobre las resenas relevantes del paso 1, extraemos campos estructurados:

1. `sentimiento_general` (positivo, negativo, mixto)
2. `aspectos_positivos` (lista de max 3 items)
3. `aspectos_negativos` (lista de max 3 items)
4. `valor_por_dinero` (alto, medio, bajo, no_menciona)
5. `dificultad_percibida` (facil, media, dificil, no_menciona)

Para este paso se usa `BATCH_SIZE_EXTRACT = 1` para minimizar truncamientos y errores de parseo en salidas JSON largas.

In [40]:
def build_extraction_prompt(batch_rows: List[Dict[str, Any]]) -> str:
    rows_json = json.dumps(batch_rows, ensure_ascii=False, indent=2)
    return f"""
Extrae informacion estructurada de cada resena de videojuegos.

Devuelve SOLO un JSON valido (lista de objetos), sin explicaciones.
Cada objeto debe contener EXACTAMENTE estas claves:
- review_id (int)
- sentimiento_general (positivo|negativo|mixto)
- aspectos_positivos (lista de strings, max 3)
- aspectos_negativos (lista de strings, max 3)
- valor_por_dinero (alto|medio|bajo|no_menciona)
- dificultad_percibida (facil|media|dificil|no_menciona)

Respuestas concisas para reducir longitud de salida:
- Cada item de aspectos debe tener maximo 8 palabras.

Si falta informacion, usa valores conservadores y listas vacias cuando proceda.

Datos de entrada:
{rows_json}
"""


def _to_list_of_strings(value: Any) -> List[str]:
    if isinstance(value, list):
        clean_values = [str(x).strip() for x in value if str(x).strip()]
        return clean_values[:3]
    if isinstance(value, str) and value.strip():
        return [value.strip()][:3]
    return []


def _normalize_extraction_item(item: Any) -> Dict[str, Any] | None:
    """Normaliza una salida del LLM para garantizar un dict utilizable."""
    if isinstance(item, str):
        item = item.strip()
        if not item:
            return None
        try:
            item = json.loads(item)
        except json.JSONDecodeError:
            return None

    if not isinstance(item, dict):
        return None

    if "review_id" not in item:
        return None

    try:
        rid = int(item["review_id"])
    except (TypeError, ValueError):
        return None

    return {
        "review_id": rid,
        "sentimiento_general": str(item.get("sentimiento_general", "mixto")),
        "aspectos_positivos": _to_list_of_strings(item.get("aspectos_positivos", [])),
        "aspectos_negativos": _to_list_of_strings(item.get("aspectos_negativos", [])),
        "valor_por_dinero": str(item.get("valor_por_dinero", "no_menciona")),
        "dificultad_percibida": str(item.get("dificultad_percibida", "no_menciona")),
    }


def _extract_review_objects_from_text(raw_text: str) -> List[Dict[str, Any]]:
    """Extrae objetos JSON individuales con review_id, incluso si la lista viene truncada."""
    text = (raw_text or "").strip()
    decoder = json.JSONDecoder()
    recovered: List[Dict[str, Any]] = []

    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and "review_id" in obj:
            normalized = _normalize_extraction_item(obj)
            if normalized is not None:
                recovered.append(normalized)

    unique: Dict[int, Dict[str, Any]] = {}
    for item in recovered:
        unique[item["review_id"]] = item
    return list(unique.values())


def run_extraction_step(
    df_input: pd.DataFrame,
    checkpoint_path: Path,
    resume: bool = True,
) -> pd.DataFrame:
    records = df_input[["review_id", "Contenido"]].to_dict(orient="records")

    # Recorte defensivo para reducir truncamientos de salida en lotes largos
    for row in records:
        text = str(row.get("Contenido", ""))
        row["Contenido"] = text[:MAX_CHARS_PER_REVIEW_EXTRACT]

    done_map: Dict[int, Dict[str, Any]] = {}
    if resume and checkpoint_path.exists():
        previous = pd.read_json(checkpoint_path, lines=True)
        for _, row in previous.iterrows():
            done_map[int(row["review_id"])] = row.to_dict()
        print(f"Checkpoint encontrado con {len(done_map)} extracciones ya procesadas.")

    pending_records = [r for r in records if int(r["review_id"]) not in done_map]
    print(f"Resenas pendientes de extraccion: {len(pending_records)}")

    system_prompt = (
        "Eres un extractor de informacion. Devuelves JSON estricto y consistente."
    )

    for batch_number, batch in enumerate(chunked(pending_records, BATCH_SIZE_EXTRACT), start=1):
        estimated_in_tokens = estimate_tokens(str(batch))
        print(
            f"Lote extraccion {batch_number}: {len(batch)} resenas | "
            f"est_tokens_in={estimated_in_tokens}"
        )

        prompt = build_extraction_prompt(batch)
        raw_response = call_llm_with_retry(
            client=client,
            model=MODEL_EXTRACT,
            system_prompt=system_prompt,
            user_prompt=prompt,
            max_tokens=MAX_TOKENS_EXTRACT,
            temperature=0.0,
        )

        parsed = []
        try:
            parsed = extract_json_array(raw_response)
        except Exception:
            parsed = []

        accepted = 0
        for item in parsed:
            normalized_item = _normalize_extraction_item(item)
            if normalized_item is None:
                continue
            rid = normalized_item["review_id"]
            done_map[rid] = normalized_item
            accepted += 1

        # Si el parser principal no consigue items validos, intentar recuperacion por objetos
        if accepted == 0:
            recovered_items = _extract_review_objects_from_text(raw_response)
            for normalized_item in recovered_items:
                rid = normalized_item["review_id"]
                done_map[rid] = normalized_item
            accepted = len(recovered_items)

        if accepted == 0:
            preview = (raw_response or "")[:500].replace("\n", " ")
            raise ValueError(
                f"No se pudo normalizar ninguna salida valida del lote. Preview: {preview}"
            )

        if accepted != len(batch):
            print(
                f"Aviso: lote con {len(batch)} entradas, se aceptaron {accepted} salidas normalizadas."
            )

        pd.DataFrame(done_map.values()).to_json(
            checkpoint_path, orient="records", lines=True, force_ascii=False
        )
        time.sleep(SLEEP_BETWEEN_CALLS)

    return pd.DataFrame(done_map.values()).sort_values("review_id")

In [41]:
extract_checkpoint = processed_dir / "checkpoint_step2_extract.jsonl"
df_extracted = run_extraction_step(df_step1_relevant, extract_checkpoint, resume=True)

df_step2 = df_step1_relevant.merge(df_extracted, on="review_id", how="left")
print("Extracciones completadas:", len(df_step2))

cols_show = [
    "review_id",
    "Valoracion",
    "sentimiento_general",
    "valor_por_dinero",
    "dificultad_percibida",
]
df_step2[cols_show].head(10)

Checkpoint encontrado con 45 extracciones ya procesadas.
Resenas pendientes de extraccion: 54
Lote extraccion 1: 1 resenas | est_tokens_in=309
Lote extraccion 2: 1 resenas | est_tokens_in=309
Lote extraccion 3: 1 resenas | est_tokens_in=309
Lote extraccion 4: 1 resenas | est_tokens_in=309
Lote extraccion 5: 1 resenas | est_tokens_in=309
Lote extraccion 6: 1 resenas | est_tokens_in=309
Lote extraccion 7: 1 resenas | est_tokens_in=309
Lote extraccion 8: 1 resenas | est_tokens_in=309
Lote extraccion 9: 1 resenas | est_tokens_in=309
Lote extraccion 10: 1 resenas | est_tokens_in=309
Lote extraccion 11: 1 resenas | est_tokens_in=309
Lote extraccion 12: 1 resenas | est_tokens_in=309
Lote extraccion 13: 1 resenas | est_tokens_in=309
Lote extraccion 14: 1 resenas | est_tokens_in=309
Lote extraccion 15: 1 resenas | est_tokens_in=309
Lote extraccion 16: 1 resenas | est_tokens_in=309
Lote extraccion 17: 1 resenas | est_tokens_in=309
Lote extraccion 18: 1 resenas | est_tokens_in=309
Lote extraccion

,review_id,Valoracion,sentimiento_general,valor_por_dinero,dificultad_percibida
0,17660,Recomendado,positivo,alto,no_menciona
1,7187,No recomendado,negativo,no_menciona,no_menciona
2,4187,No recomendado,negativo,no_menciona,no_menciona
3,7751,No recomendado,mixto,no_menciona,no_menciona
4,5415,No recomendado,negativo,no_menciona,dificil
5,16503,Recomendado,negativo,no_menciona,no_menciona
6,7805,No recomendado,negativo,no_menciona,no_menciona
7,13941,Recomendado,positivo,no_menciona,dificil
8,12814,Recomendado,positivo,no_menciona,dificil
9,13993,Recomendado,positivo,no_menciona,no_menciona


In [42]:
# Analisis basico de resultados estructurados
print("Distribucion de sentimiento_general:")
print(df_step2["sentimiento_general"].value_counts(dropna=False))

print("\nCruce sentimiento_general vs Valoracion:")
display(pd.crosstab(df_step2["sentimiento_general"], df_step2["Valoracion"], margins=True))

print("\nEjemplo de aspectos positivos/negativos:")
df_step2[["review_id", "aspectos_positivos", "aspectos_negativos"]].head(5)

Distribucion de sentimiento_general:
sentimiento_general
mixto       44
negativo    35
positivo    20
Name: count, dtype: int64

Cruce sentimiento_general vs Valoracion:


Valoracion,No recomendado,Recomendado,All
sentimiento_general,,,
mixto,32,12,44
negativo,32,3,35
positivo,1,19,20
All,65,34,99



Ejemplo de aspectos positivos/negativos:


,review_id,aspectos_positivos,aspectos_negativos
0,17660,"[Diseño y detalle de Night City, Ambientación ...",[Problemas técnicos menores]
1,7187,[],"[Estado general decepcionante, Presencia de bu..."
2,4187,[],"[Estado general decepcionante, Presencia de bu..."
3,7751,"[Combate fluido y controles responsivos, Anima...","[Decisiones de diseño poco meditadas, Reutiliz..."
4,5415,"[ambientacion hermosa, historia fascinante, pe...","[dificultad excesivamente punitiva, ataques en..."


In [43]:
# Guardado final de salidas
step2_json_path = processed_dir / "ej1_parte2_analisis.json"
entities_json_path = processed_dir / "extracted_entities.json"

df_step2.to_json(step2_json_path, orient="records", force_ascii=False, indent=2)
df_step2[[
    "review_id",
    "sentimiento_general",
    "aspectos_positivos",
    "aspectos_negativos",
    "valor_por_dinero",
    "dificultad_percibida",
]].to_json(entities_json_path, orient="records", force_ascii=False, indent=2)

print(f"Guardado Paso 2 en: {step2_json_path}")
print(f"Entidades extraidas en: {entities_json_path}")

Guardado Paso 2 en: C:\Users\venel\Documents\GitHub\IA0526\PRACTICA2_PROMPT\data\processed\ej1_parte2_analisis.json
Entidades extraidas en: C:\Users\venel\Documents\GitHub\IA0526\PRACTICA2_PROMPT\data\processed\extracted_entities.json


## Conclusiones y control de limitaciones

- Se implemento un pipeline LLM de 2 pasos: filtro + extraccion.
- Se justifico usar Gemini por latencia/coste/calidad para lotes.
- Se parametrizo el tamano de lotes por paso (`BATCH_SIZE_FILTER`, `BATCH_SIZE_EXTRACT`).
- Se aplico pausa entre llamadas (`SLEEP_BETWEEN_CALLS`) para respetar limites de uso.
- Se aplicaron reintentos con backoff ante errores transitorios (`MAX_RETRIES`, `BACKOFF_BASE_SECONDS`).
- Se implementaron checkpoints para reanudar ejecucion si se corta por limite/cuota/red.

### Si se alcanza un limite antes de terminar

1. Esperar la ventana de rate limit del proveedor.
2. Re-ejecutar las celdas del paso correspondiente.
3. El notebook continua desde checkpoint y solo procesa pendientes.